# Notebook 05 — LSTM Autoencoder Anomaly Scoring (All 10 Splits)


Trains one LSTM Autoencoder per random split on **clean normal-only windows**, scores every row (train / val / test) with the per-step mean reconstruction error, and saves results alongside per-split plots and metrics.

**Run order:** after preprocessing (02b), before 05e (architecture comparison).

## 1. Imports

In [1]:
from pathlib import Path
import gc
import random

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    roc_curve,
)

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# Device
if torch.backends.mps.is_available():
    device = torch.device("mps")
    torch.mps.manual_seed(SEED)
elif torch.cuda.is_available():
    device = torch.device("cuda")
    torch.cuda.manual_seed_all(SEED)
else:
    device = torch.device("cpu")

print("Using device:", device)

Using device: mps


## 2. Paths

In [2]:


PROJECT_DIR = Path("/Users/arjun/Documents/UAV thesis/uav-thesis/uav_px4_dataset_project")


DATA_VERSION = "v5_voltage_random10_new_features"


INPUT_ROOT        = PROJECT_DIR / f"model_ready_no_dr_{DATA_VERSION}"  
EXPERIMENT_NAME   = f"lstm_autoencoder_{DATA_VERSION}"
MASTER_OUTPUT_DIR = PROJECT_DIR / "model_outputs" / EXPERIMENT_NAME

MASTER_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Input root :", INPUT_ROOT)
print("Output root:", MASTER_OUTPUT_DIR)

Input root : /Users/arjun/Documents/UAV thesis/uav-thesis/uav_px4_dataset_project/model_ready_no_dr_v5_voltage_random10_new_features
Output root: /Users/arjun/Documents/UAV thesis/uav-thesis/uav_px4_dataset_project/model_outputs/lstm_autoencoder_v5_voltage_random10_new_features


## 3. Splits

In [3]:


SPLIT_SEEDS = [11, 22, 33, 44, 55, 66, 77, 88, 99, 111]

ALL_SPLITS = [
    f"split_{str(i + 1).zfill(2)}_seed_{seed}"
    for i, seed in enumerate(SPLIT_SEEDS)
]

print("\nSplits to run:")
for s in ALL_SPLITS:
    print(" ", s)


Splits to run:
  split_01_seed_11
  split_02_seed_22
  split_03_seed_33
  split_04_seed_44
  split_05_seed_55
  split_06_seed_66
  split_07_seed_77
  split_08_seed_88
  split_09_seed_99
  split_10_seed_111


## 4. Config

In [4]:

target_col = "target_energy_next_60s_wh"

feature_cols = [
    "battery_status__cell_count",
    "battery_status__current_a",
    "battery_status__current_filtered_a",
    "battery_status__discharged_mah",
    "battery_status__remaining",
    "battery_status__scale",
    "battery_status__voltage_filtered_v",
    "battery_status__voltage_v",
    "cumulative_energy_wh",
    "power_w",
    "time_s",
    "is_idle",
    "power_w_rolling_std_3s",
    "voltage_per_cell",
    "voltage_filtered_per_cell",
    "voltage_per_cell_suspicious",
]

assert len(feature_cols) == 16
assert "energy_increment_wh" not in feature_cols

SEQ_LEN       = 100   # 10 seconds at 10 Hz
STRIDE_TRAIN  = 20    # 2-second stride — overlapping windows during training
STRIDE_SCORE  = 10    # 1-second stride — denser coverage during scoring
BATCH_SIZE    = 64
EPOCHS        = 200
PATIENCE      = 15
MAX_GRAD_NORM = 1.0
MIN_DELTA     = 1e-5
FORCE_RERUN   = False  #must set True to overwrite already-completed splits

print(f"\nFeatures  : {len(feature_cols)}")
print(f"Seq len   : {SEQ_LEN} rows = {SEQ_LEN / 10:.0f}s at 10 Hz")
print(f"FORCE_RERUN: {FORCE_RERUN}")


Features  : 16
Seq len   : 100 rows = 10s at 10 Hz
FORCE_RERUN: False


## 5.  Data helpers

In [5]:

def load_split_files(input_dir: Path, split_name: str) -> tuple:
    """
    Load the train, validation, and test parquet files for one split.

    Args:
        input_dir (Path): directory that contains the three parquet
            files for this split.
        split_name (str): split identifier, e.g. 'split_05_seed_55'.

    Returns:
        tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]: the train,
            validation, and test DataFrames in that order.

    Raises:
        FileNotFoundError: if any of the three files does not exist.
    """
    paths = {
        "train": input_dir / f"train_no_dr_scaled_{DATA_VERSION}_{split_name}.parquet",
        "val":   input_dir / f"val_no_dr_scaled_{DATA_VERSION}_{split_name}.parquet",
        "test":  input_dir / f"test_no_dr_scaled_{DATA_VERSION}_{split_name}.parquet",
    }
    for name, path in paths.items():
        if not path.exists():
            raise FileNotFoundError(f"Missing {name} file for {split_name}:\n{path}")

    train_df = pd.read_parquet(paths["train"])
    val_df   = pd.read_parquet(paths["val"])
    test_df  = pd.read_parquet(paths["test"])

    print("Loaded files:")
    print(f"  Train: {paths['train'].name}")
    print(f"  Val  : {paths['val'].name}")
    print(f"  Test : {paths['test'].name}")

    return train_df, val_df, test_df


def validate_data(
    train_df:   pd.DataFrame,
    val_df:     pd.DataFrame,
    test_df:    pd.DataFrame,
    split_name: str,
) -> None:
    """
    Verify that all required columns are present and that no NaN
    values exist in any feature column before training begins.

    Args:
        train_df (pd.DataFrame): training split DataFrame.
        val_df (pd.DataFrame): validation split DataFrame.
        test_df (pd.DataFrame): test split DataFrame.
        split_name (str): used in error messages for clarity.

    Raises:
        ValueError: if a required column is missing or a feature
            column contains NaN values.
    """
    required_cols = (
        ["log_id", "dataset_label", target_col]
        + feature_cols
    )
    for label, df in [("Train", train_df), ("Validation", val_df), ("Test", test_df)]:
        missing = [c for c in required_cols if c not in df.columns]
        if missing:
            raise ValueError(f"{split_name} | {label}: missing columns: {missing}")

        nan_count = int(df[feature_cols].isna().sum().sum())
        if nan_count > 0:
            bad = df[feature_cols].columns[df[feature_cols].isna().any()].tolist()
            raise ValueError(
                f"{split_name} | {label}: {nan_count} NaN values in: {bad}"
            )
    print("Data validation passed.")


def prepare_sorted_df(df: pd.DataFrame) -> pd.DataFrame:
    """
    Sort a split DataFrame by log_id then time_s and tag each row
    with its original position so outputs can be restored to the
    original file order before saving.

    Args:
        df (pd.DataFrame): raw split DataFrame (unsorted).

    Returns:
        pd.DataFrame: sorted copy with an added '_original_order'
            integer column and a clean 0-based integer index.
    """
    df = df.copy()
    df["_original_order"] = np.arange(len(df))
    df["log_id"]          = df["log_id"].astype(str)
    return df.sort_values(["log_id", "time_s"]).reset_index(drop=True)


def build_window_starts(
    df_sorted:   pd.DataFrame,
    seq_len:     int,
    stride:      int,
    normal_only: bool = False,
    clean_only:  bool = False,
) -> np.ndarray:
    """
    Compute the start row index for every valid sliding window across
    all flights.  Windows never cross flight boundaries.

    Args:
        df_sorted (pd.DataFrame): DataFrame sorted by log_id / time_s
            with a clean 0-based integer index.
        seq_len (int): number of rows per window.
        stride (int): step size between consecutive window starts.
        normal_only (bool): when True, skip any window that contains
            a row labelled 'abnormal'.  Used during training so the
            autoencoder learns only from normal behaviour.
        clean_only (bool): when True, skip windows where any row is
            flagged as voltage_per_cell_suspicious.  Requires the
            column 'voltage_per_cell_suspicious' to be present.

    Returns:
        np.ndarray: 1-D int64 array of start row indices into df_sorted.
    """
    window_starts = []

    for _, group in df_sorted.groupby("log_id", sort=False):
        indices = group.index.to_numpy()
        n       = len(group)

        if n < seq_len:
            continue

        for local_start in range(0, n - seq_len + 1, stride):
            win_idx = indices[local_start : local_start + seq_len]
            win_df  = df_sorted.loc[win_idx]

            if normal_only and not (win_df["dataset_label"] == "normal").all():
                continue

            if clean_only and not (win_df["voltage_per_cell_suspicious"] == 0).all():
                continue

            window_starts.append(win_idx[0])

    return np.array(window_starts, dtype=np.int64)


def restore_and_save(scored_df: pd.DataFrame, save_path: Path) -> pd.DataFrame:
    """
    Restore the original row order, drop the helper column added by
    prepare_sorted_df(), and save the result as a CSV.

    Args:
        scored_df (pd.DataFrame): scored DataFrame that still contains
            the '_original_order' column.
        save_path (Path): destination path for the CSV file.

    Returns:
        pd.DataFrame: the saved DataFrame in original row order.
    """
    output_df = (
        scored_df
        .sort_values("_original_order")
        .drop(columns=["_original_order"])
        .reset_index(drop=True)
    )
    output_df.to_csv(save_path, index=False)
    print(f"Saved : {save_path}")
    print(f"Shape : {output_df.shape}")
    print(f"Missing anomaly_score: {output_df['anomaly_score'].isna().sum()}")
    return output_df

## 6.  Dataset classes

In [6]:


class UAVTrainSequenceDataset(Dataset):
    """
    Yields fixed-length input sequences for autoencoder training.

    Only the sequence tensor is returned because the row index is
    not needed during the forward / backward pass.

    Args:
        data_array (np.ndarray): 2-D float32 array of shape
            (n_rows, n_features) — the full sorted split matrix.
        window_starts (np.ndarray): 1-D int64 array of start indices
            produced by build_window_starts().
        seq_len (int): number of time steps per window.
    """

    def __init__(
        self,
        data_array:    np.ndarray,
        window_starts: np.ndarray,
        seq_len:       int,
    ):
        self.data_array    = data_array
        self.window_starts = window_starts
        self.seq_len       = seq_len

    def __len__(self) -> int:
        return len(self.window_starts)

    def __getitem__(self, idx: int) -> torch.Tensor:
        """
        Args:
            idx (int): window index.

        Returns:
            torch.Tensor: float32 tensor of shape (seq_len, n_features).
        """
        start = self.window_starts[idx]
        return torch.tensor(
            self.data_array[start : start + self.seq_len],
            dtype=torch.float32,
        )


class UAVScoringSequenceDataset(Dataset):
    """
    Yields (sequence, start_index) pairs for anomaly scoring.

    The start index is returned alongside the sequence so per-step
    reconstruction errors can be accumulated back into the correct
    rows of the output DataFrame.

    Args:
        data_array (np.ndarray): 2-D float32 array of shape
            (n_rows, n_features).
        window_starts (np.ndarray): 1-D int64 array of start indices.
        seq_len (int): number of time steps per window.
    """

    def __init__(
        self,
        data_array:    np.ndarray,
        window_starts: np.ndarray,
        seq_len:       int,
    ):
        self.data_array    = data_array
        self.window_starts = window_starts
        self.seq_len       = seq_len

    def __len__(self) -> int:
        return len(self.window_starts)

    def __getitem__(self, idx: int) -> tuple:
        """
        Args:
            idx (int): window index.

        Returns:
            tuple[torch.Tensor, int]: the sequence tensor of shape
                (seq_len, n_features) and its start row index into
                the full DataFrame.
        """
        start = self.window_starts[idx]
        return (
            torch.tensor(
                self.data_array[start : start + self.seq_len],
                dtype=torch.float32,
            ),
            start,
        )

## 7. Model

In [7]:


class LSTMAutoencoder(nn.Module):
    """
    LSTM sequence autoencoder for UAV battery anomaly detection.

    The encoder reads the full sequence and compresses it to a latent
    vector via the final hidden state.  The decoder receives that
    latent vector repeated across all time steps as its input and is
    also initialised with learned h0 / c0 projections, forcing the
    model to reconstruct the entire sequence from a compact bottleneck
    (Malhotra et al. 2016).

    Per-step MSE reconstruction error serves as the anomaly score:
    flights that deviate from normal battery behaviour produce higher
    errors than normal ones.

    Args:
        input_size (int): number of input features per time step.
        hidden_size (int): LSTM hidden dimension for both encoder and
            decoder.  Defaults to 64.
        latent_size (int): bottleneck dimension between the encoder FC
            and the decoder FC.  Defaults to 32.
        num_layers (int): number of stacked LSTM layers.  Defaults to 1.
    """

    def __init__(
        self,
        input_size:  int,
        hidden_size: int = 64,
        latent_size: int = 32,
        num_layers:  int = 1,
    ):
        super().__init__()

        self.input_size  = input_size
        self.hidden_size = hidden_size
        self.latent_size = latent_size
        self.num_layers  = num_layers

        self.encoder_lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
        )
        self.encoder_fc = nn.Linear(hidden_size, latent_size)

        self.decoder_input_fc = nn.Linear(latent_size, hidden_size)
        self.decoder_h0_fc    = nn.Linear(latent_size, hidden_size * num_layers)
        self.decoder_c0_fc    = nn.Linear(latent_size, hidden_size * num_layers)
        self.decoder_lstm     = nn.LSTM(
            input_size=hidden_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
        )
        self.output_layer = nn.Linear(hidden_size, input_size)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Encode the input sequence to a latent vector and reconstruct it.

        Args:
            x (torch.Tensor): input tensor of shape
                (batch_size, seq_len, input_size).

        Returns:
            torch.Tensor: reconstructed tensor of the same shape as x.
        """
        batch_size, seq_len, _ = x.shape

        _, (hidden, _) = self.encoder_lstm(x)
        hidden_last    = hidden[-1]
        latent         = self.encoder_fc(hidden_last)

        decoder_input = (
            self.decoder_input_fc(latent)
            .unsqueeze(1)
            .repeat(1, seq_len, 1)
        )

        h0 = (
            self.decoder_h0_fc(latent)
            .view(batch_size, self.num_layers, self.hidden_size)
            .permute(1, 0, 2)
            .contiguous()
        )
        c0 = (
            self.decoder_c0_fc(latent)
            .view(batch_size, self.num_layers, self.hidden_size)
            .permute(1, 0, 2)
            .contiguous()
        )

        decoder_output, _ = self.decoder_lstm(decoder_input, (h0, c0))
        return self.output_layer(decoder_output)

## 8. Scoring function

In [8]:


def score_dataframe_with_autoencoder(
    df_sorted:      pd.DataFrame,
    feature_cols:   list,
    seq_len:        int,
    stride:         int,
    model_to_score: nn.Module,
    batch_size:     int = 64,
    device:         torch.device = device,
) -> pd.DataFrame:
    """
    Assign a per-row anomaly score to every row in df_sorted.

    Each row's score is the average per-step MSE reconstruction error
    across all windows that include that row.  Rows not covered by any
    window (short tail rows in brief flights) are filled first by
    forward / back-fill within the same flight, then by the global
    normal-row median as a last resort.

    Args:
        df_sorted (pd.DataFrame): DataFrame sorted by log_id / time_s
            with a clean 0-based integer index.
        feature_cols (list[str]): column names passed to the model as
            input features.
        seq_len (int): window length in rows.
        stride (int): step between consecutive window starts during
            scoring.
        model_to_score (nn.Module): trained model instance.  Should
            already be in eval mode on the correct device.
        batch_size (int): inference batch size.  Defaults to 64.
        device (torch.device): device to move batches to.

    Returns:
        pd.DataFrame: copy of df_sorted with an 'anomaly_score' column
            added.  The column will contain no NaN values.
    """
    model_to_score.eval()

    X_array = df_sorted[feature_cols].to_numpy(dtype=np.float32)

    window_starts = build_window_starts(
        df_sorted,
        seq_len=seq_len,
        stride=stride,
        normal_only=False,
        clean_only=False,
    )
    print(f"  Scoring windows: {len(window_starts):,}")

    dataset = UAVScoringSequenceDataset(X_array, window_starts, seq_len)
    loader  = DataLoader(dataset, batch_size=batch_size, shuffle=False, drop_last=False)

    score_sum   = np.zeros(len(df_sorted), dtype=np.float64)
    score_count = np.zeros(len(df_sorted), dtype=np.float64)

    with torch.no_grad():
        for batch_x, batch_starts in loader:
            batch_x       = batch_x.to(device)
            reconstructed = model_to_score(batch_x)

            step_errors  = ((batch_x - reconstructed) ** 2).mean(dim=2)
            step_errors  = step_errors.cpu().numpy()
            batch_starts = batch_starts.numpy()

            for i, start in enumerate(batch_starts):
                end = start + seq_len
                score_sum[start:end]   += step_errors[i]
                score_count[start:end] += 1

    anomaly_score = np.where(score_count > 0, score_sum / score_count, np.nan)

    scored_df                  = df_sorted.copy()
    scored_df["anomaly_score"] = anomaly_score

    uncovered = int(scored_df["anomaly_score"].isna().sum())
    if uncovered > 0:
        print(f"  Uncovered rows: {uncovered:,} — filling with ffill/bfill within flights")
        scored_df["anomaly_score"] = (
            scored_df
            .groupby("log_id")["anomaly_score"]
            .transform(lambda s: s.ffill().bfill())
        )

    still_missing = int(scored_df["anomaly_score"].isna().sum())
    if still_missing > 0:
        normal_median = scored_df.loc[
            scored_df["dataset_label"] == "normal", "anomaly_score"
        ].median()
        fallback = normal_median if pd.notna(normal_median) else scored_df["anomaly_score"].median()
        scored_df["anomaly_score"] = scored_df["anomaly_score"].fillna(fallback)
        print(f"  Global fallback applied to {still_missing:,} rows → {fallback:.6f}")

    return scored_df

## 9. Summary helpers

In [9]:


def create_row_summary(
    df:         pd.DataFrame,
    split_name: str,
    data_split: str,
) -> pd.DataFrame:
    """
    Compute row-level anomaly score statistics broken down by label.

    Args:
        df (pd.DataFrame): scored DataFrame containing 'dataset_label',
            'anomaly_score', 'anomaly_flag', and all signal columns.
        split_name (str): split identifier, e.g. 'split_05_seed_55'.
        data_split (str): one of 'Train', 'Validation', 'Test'.

    Returns:
        pd.DataFrame: one row per present label with mean, std, median,
            and 95th-percentile score plus key signal means.
    """
    records = []
    for label in ["normal", "abnormal"]:
        subset = df[df["dataset_label"] == label]
        if len(subset) == 0:
            continue
        records.append({
            "split":                 split_name,
            "data_split":            data_split,
            "dataset_label":         label,
            "n_rows":                len(subset),
            "mean_score":            subset["anomaly_score"].mean(),
            "std_score":             subset["anomaly_score"].std(),
            "median_score":          subset["anomaly_score"].median(),
            "p95_score":             subset["anomaly_score"].quantile(0.95),
            "flagged_pct":           subset["anomaly_flag"].mean() * 100,
            "mean_power_w":          subset["power_w"].mean(),
            "mean_current":          subset["battery_status__current_a"].mean(),
            "mean_voltage":          subset["battery_status__voltage_filtered_v"].mean(),
            "mean_remaining":        subset["battery_status__remaining"].mean(),
            "mean_voltage_per_cell": subset["voltage_per_cell"].mean(),
            "suspicious_pct":        subset["voltage_per_cell_suspicious"].mean() * 100,
        })
    return pd.DataFrame(records)


def create_flight_summary(
    df:         pd.DataFrame,
    split_name: str,
    data_split: str,
    threshold:  float,
) -> pd.DataFrame:
    """
    Aggregate scored row-level data to one row per flight.

    Args:
        df (pd.DataFrame): scored DataFrame with 'anomaly_score',
            'anomaly_flag', and all signal columns present.
        split_name (str): split identifier.
        data_split (str): one of 'Train', 'Validation', 'Test'.
        threshold (float): anomaly threshold value stored as a column
            so downstream notebooks can read it back without recomputing.

    Returns:
        pd.DataFrame: per-flight statistics including mean / max / std
            score, flagged percentage, key signal means, voltage quality
            indicators, and split metadata.
    """
    flight_summary = (
        df.groupby(["log_id", "dataset_label"])
        .agg(
            mean_anomaly_score            =("anomaly_score",                     "mean"),
            max_anomaly_score             =("anomaly_score",                     "max"),
            std_anomaly_score             =("anomaly_score",                     "std"),
            flagged_percentage            =("anomaly_flag",                      "mean"),
            rows                          =("anomaly_score",                     "size"),
            mean_power_w                  =("power_w",                           "mean"),
            mean_current                  =("battery_status__current_a",         "mean"),
            mean_voltage                  =("battery_status__voltage_filtered_v","mean"),
            mean_remaining                =("battery_status__remaining",         "mean"),
            mean_voltage_per_cell         =("voltage_per_cell",                  "mean"),
            mean_voltage_filtered_per_cell=("voltage_filtered_per_cell",         "mean"),
            suspicious_percentage         =("voltage_per_cell_suspicious",       "mean"),
        )
        .reset_index()
    )

    flight_summary["flagged_percentage"]    *= 100
    flight_summary["suspicious_percentage"] *= 100
    flight_summary["split"]                  = split_name
    flight_summary["data_split"]             = data_split
    flight_summary["threshold"]              = threshold

    return flight_summary


def compute_flight_metrics(
    flight_summary: pd.DataFrame,
    split_name:     str,
    data_split:     str,
) -> dict:
    """
    Compute flight-level anomaly detection metrics from the per-flight
    summary table.

    Args:
        flight_summary (pd.DataFrame): per-flight DataFrame with columns
            'dataset_label', 'mean_anomaly_score', and
            'flagged_percentage'.
        split_name (str): split identifier.
        data_split (str): one of 'Train', 'Validation', 'Test'.

    Returns:
        dict: keys are split, data_split, n_normal_flights,
            n_abnormal_flights, mean_flight_score_normal,
            mean_flight_score_abnormal, separation_ratio,
            mean_flagged_pct_normal, mean_flagged_pct_abnormal,
            flight_auroc, flight_auprc, baseline_auprc.
            AUROC / AUPRC are np.nan when only one class is present.
    """
    flight_eval = flight_summary.copy()
    flight_eval["binary_label"] = (
        flight_eval["dataset_label"] == "abnormal"
    ).astype(int)

    n_normal   = int((flight_eval["binary_label"] == 0).sum())
    n_abnormal = int((flight_eval["binary_label"] == 1).sum())

    normal_flights   = flight_summary[flight_summary["dataset_label"] == "normal"]
    abnormal_flights = flight_summary[flight_summary["dataset_label"] == "abnormal"]

    normal_mean   = normal_flights["mean_anomaly_score"].mean()
    abnormal_mean = abnormal_flights["mean_anomaly_score"].mean()

    separation_ratio = (
        abnormal_mean / normal_mean
        if pd.notna(abnormal_mean) and normal_mean > 0
        else np.nan
    )

    if flight_eval["binary_label"].nunique() == 2:
        auroc = roc_auc_score(
            flight_eval["binary_label"],
            flight_eval["mean_anomaly_score"],
        )
        auprc = average_precision_score(
            flight_eval["binary_label"],
            flight_eval["mean_anomaly_score"],
        )
        baseline_auprc = flight_eval["binary_label"].mean()
    else:
        auroc = auprc = baseline_auprc = np.nan

    return {
        "split":                      split_name,
        "data_split":                 data_split,
        "n_normal_flights":           n_normal,
        "n_abnormal_flights":         n_abnormal,
        "mean_flight_score_normal":   normal_mean,
        "mean_flight_score_abnormal": abnormal_mean,
        "separation_ratio":           separation_ratio,
        "mean_flagged_pct_normal":    normal_flights["flagged_percentage"].mean(),
        "mean_flagged_pct_abnormal":  abnormal_flights["flagged_percentage"].mean(),
        "flight_auroc":               auroc,
        "flight_auprc":               auprc,
        "baseline_auprc":             baseline_auprc,
    }

## 10. Plot helpers

In [10]:

def save_training_curve(
    train_losses: list,
    val_losses:   list,
    plot_dir:     Path,
    split_name:   str,
) -> None:
    """
    Plot and save the train / validation reconstruction loss curves.

    Args:
        train_losses (list[float]): per-epoch training MSE values.
        val_losses (list[float]): per-epoch validation MSE values.
        plot_dir (Path): directory to write the PNG into.
        split_name (str): used in the chart title and filename.
    """
    plt.figure(figsize=(8, 5))
    plt.plot(range(1, len(train_losses) + 1), train_losses, label="Train loss")
    plt.plot(range(1, len(val_losses)   + 1), val_losses,   label="Validation loss")
    plt.xlabel("Epoch")
    plt.ylabel("Reconstruction loss")
    plt.title(f"LSTM AE Training Curve — {split_name}")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()

    save_path = plot_dir / f"training_curve_{DATA_VERSION}_{split_name}.png"
    plt.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.close()
    print("Saved plot:", save_path)


def save_distribution_plot(
    train_out:  pd.DataFrame,
    val_out:    pd.DataFrame,
    test_out:   pd.DataFrame,
    threshold:  float,
    plot_dir:   Path,
    split_name: str,
) -> None:
    """
    Save a 3-panel anomaly score distribution plot — one panel per
    data split — with normal and abnormal histograms overlaid and the
    anomaly threshold marked as a vertical line.

    Args:
        train_out (pd.DataFrame): scored training DataFrame.
        val_out (pd.DataFrame): scored validation DataFrame.
        test_out (pd.DataFrame): scored test DataFrame.
        threshold (float): value at which a row is flagged as anomalous.
        plot_dir (Path): directory to write the PNG into.
        split_name (str): used in the suptitle and filename.
    """
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    for ax, (data_split, df) in zip(
        axes,
        [("Train", train_out), ("Validation", val_out), ("Test", test_out)],
    ):
        for label, color in [("normal", "steelblue"), ("abnormal", "tomato")]:
            subset = df[df["dataset_label"] == label]
            if len(subset) > 0:
                ax.hist(
                    subset["anomaly_score"],
                    bins=60,
                    alpha=0.6,
                    density=True,
                    label=f"{label} (n={len(subset):,})",
                    color=color,
                )

        ax.axvline(
            threshold,
            color="black",
            linestyle="--",
            linewidth=1.2,
            label=f"Threshold={threshold:.4f}",
        )
        ax.set_title(data_split)
        ax.set_xlabel("Anomaly score")
        ax.set_ylabel("Density")
        ax.legend(fontsize=8)
        ax.grid(True, alpha=0.3)

    plt.suptitle(f"Anomaly Score Distribution — {split_name}", fontsize=13, y=1.02)
    plt.tight_layout()

    save_path = plot_dir / f"anomaly_score_distribution_{DATA_VERSION}_{split_name}.png"
    plt.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.close()
    print("Saved plot:", save_path)


def save_flight_bar_plot(
    flight_summary: pd.DataFrame,
    plot_dir:       Path,
    split_name:     str,
    data_split:     str,
) -> None:
    """
    Save a bar chart of per-flight mean anomaly scores, ranked highest
    to lowest and coloured by normal / abnormal label.

    Args:
        flight_summary (pd.DataFrame): per-flight summary DataFrame
            with 'mean_anomaly_score' and 'dataset_label' columns.
        plot_dir (Path): directory to write the PNG into.
        split_name (str): used in the title and filename.
        data_split (str): 'Validation' or 'Test' — included in the title.
    """
    plot_df = (
        flight_summary
        .sort_values("mean_anomaly_score", ascending=False)
        .reset_index(drop=True)
    )
    colors = plot_df["dataset_label"].map({"normal": "steelblue", "abnormal": "tomato"})

    plt.figure(figsize=(14, 5))
    plt.bar(range(len(plot_df)), plot_df["mean_anomaly_score"], color=colors)
    plt.xlabel("Flight ranked by mean anomaly score")
    plt.ylabel("Mean anomaly score")
    plt.title(f"Per-Flight Mean Anomaly Score — {split_name} — {data_split}")
    plt.grid(True, alpha=0.3, axis="y")

    legend_items = [
        mpatches.Patch(facecolor="steelblue", label="normal"),
        mpatches.Patch(facecolor="tomato",    label="abnormal"),
    ]
    plt.legend(handles=legend_items)
    plt.tight_layout()

    save_path = plot_dir / f"per_flight_score_{data_split.lower()}_{DATA_VERSION}_{split_name}.png"
    plt.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.close()
    print("Saved plot:", save_path)


def save_roc_plot(
    flight_summary: pd.DataFrame,
    plot_dir:       Path,
    split_name:     str,
    data_split:     str,
) -> None:
    """
    Save a flight-level ROC curve.  The function skips silently when
    only one class is present in the split (no AUROC is meaningful).

    Args:
        flight_summary (pd.DataFrame): per-flight summary DataFrame
            with 'dataset_label' and 'mean_anomaly_score'.
        plot_dir (Path): directory to write the PNG into.
        split_name (str): used in the title and filename.
        data_split (str): 'Validation' or 'Test'.
    """
    flight_eval = flight_summary.copy()
    flight_eval["binary_label"] = (
        flight_eval["dataset_label"] == "abnormal"
    ).astype(int)

    if flight_eval["binary_label"].nunique() < 2:
        print(f"Skipping ROC plot for {split_name} {data_split}: only one class.")
        return

    fpr, tpr, _ = roc_curve(
        flight_eval["binary_label"],
        flight_eval["mean_anomaly_score"],
    )
    auroc = roc_auc_score(
        flight_eval["binary_label"],
        flight_eval["mean_anomaly_score"],
    )

    plt.figure(figsize=(6, 5))
    plt.plot(fpr, tpr, label=f"LSTM AE AUROC={auroc:.4f}")
    plt.plot([0, 1], [0, 1], linestyle="--", color="gray", label="Random")
    plt.xlabel("False positive rate")
    plt.ylabel("True positive rate")
    plt.title(f"ROC Curve — {split_name} — {data_split}")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()

    save_path = plot_dir / f"roc_curve_{data_split.lower()}_{DATA_VERSION}_{split_name}.png"
    plt.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.close()
    print("Saved plot:", save_path)

## 11. MAIN LOOP

In [11]:


ALL_METRICS          = []
ALL_ROW_SUMMARIES    = []
ALL_FLIGHT_SUMMARIES = []

for SPLIT_NAME in ALL_SPLITS:

    print("\n" + "=" * 100)
    print(f"STARTING SPLIT: {SPLIT_NAME}")
    print("=" * 100)

    INPUT_DIR  = INPUT_ROOT / SPLIT_NAME
    OUTPUT_DIR = MASTER_OUTPUT_DIR / SPLIT_NAME
    PLOT_DIR   = OUTPUT_DIR / "plots"

    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    PLOT_DIR.mkdir(parents=True, exist_ok=True)

    model_path = OUTPUT_DIR / "best_lstm_autoencoder.pt"

    train_save_path = OUTPUT_DIR / f"train_no_dr_with_lstm_anomaly_score_{DATA_VERSION}_{SPLIT_NAME}.csv"
    val_save_path   = OUTPUT_DIR / f"val_no_dr_with_lstm_anomaly_score_{DATA_VERSION}_{SPLIT_NAME}.csv"
    test_save_path  = OUTPUT_DIR / f"test_no_dr_with_lstm_anomaly_score_{DATA_VERSION}_{SPLIT_NAME}.csv"

    train_flight_summary_path = OUTPUT_DIR / f"flight_summary_train_{DATA_VERSION}_{SPLIT_NAME}.csv"
    val_flight_summary_path   = OUTPUT_DIR / f"flight_summary_validation_{DATA_VERSION}_{SPLIT_NAME}.csv"
    test_flight_summary_path  = OUTPUT_DIR / f"flight_summary_test_{DATA_VERSION}_{SPLIT_NAME}.csv"

    row_summary_path = OUTPUT_DIR / f"row_summary_{DATA_VERSION}_{SPLIT_NAME}.csv"
    metrics_path     = OUTPUT_DIR / f"metrics_{DATA_VERSION}_{SPLIT_NAME}.csv"
    history_path     = OUTPUT_DIR / f"training_history_{DATA_VERSION}_{SPLIT_NAME}.csv"

    completed = (
        train_save_path.exists()
        and val_save_path.exists()
        and test_save_path.exists()
        and test_flight_summary_path.exists()
    )

    if completed and not FORCE_RERUN:
        print("Split already completed. Loading saved outputs.")

        train_out            = pd.read_csv(train_save_path,           low_memory=False)
        val_out              = pd.read_csv(val_save_path,             low_memory=False)
        test_out             = pd.read_csv(test_save_path,            low_memory=False)
        train_flight_summary = pd.read_csv(train_flight_summary_path, low_memory=False)
        val_flight_summary   = pd.read_csv(val_flight_summary_path,   low_memory=False)
        test_flight_summary  = pd.read_csv(test_flight_summary_path,  low_memory=False)
        threshold            = float(test_flight_summary["threshold"].iloc[0])

    else:
        train_df, val_df, test_df = load_split_files(INPUT_DIR, SPLIT_NAME)

        print("Train:", train_df.shape)
        print("Val  :", val_df.shape)
        print("Test :", test_df.shape)

        validate_data(train_df, val_df, test_df, SPLIT_NAME)

        train_sorted = prepare_sorted_df(train_df)
        val_sorted   = prepare_sorted_df(val_df)
        test_sorted  = prepare_sorted_df(test_df)

        # Training uses clean normal windows only
        train_window_starts = build_window_starts(
            train_sorted,
            seq_len=SEQ_LEN,
            stride=STRIDE_TRAIN,
            normal_only=True,
        )
        val_ae_window_starts = build_window_starts(
            val_sorted,
            seq_len=SEQ_LEN,
            stride=STRIDE_TRAIN,
            normal_only=True,
        )

        print(f"Clean normal training windows   : {len(train_window_starts):,}")
        print(f"Clean normal validation windows : {len(val_ae_window_starts):,}")

        if len(train_window_starts) == 0:
            print(f"Skipping {SPLIT_NAME}: no clean normal training windows.")
            continue
        if len(val_ae_window_starts) == 0:
            print(f"Skipping {SPLIT_NAME}: no clean normal validation windows.")
            continue

        X_train_array = train_sorted[feature_cols].to_numpy(dtype=np.float32)
        X_val_array   = val_sorted[feature_cols].to_numpy(dtype=np.float32)

        train_dataset = UAVTrainSequenceDataset(X_train_array, train_window_starts,   SEQ_LEN)
        val_ae_dataset = UAVTrainSequenceDataset(X_val_array,  val_ae_window_starts,  SEQ_LEN)

        train_loader  = DataLoader(train_dataset,  batch_size=BATCH_SIZE, shuffle=True,  drop_last=False)
        val_ae_loader = DataLoader(val_ae_dataset, batch_size=BATCH_SIZE, shuffle=False, drop_last=False)

        print("Train batches     :", len(train_loader))
        print("Validation batches:", len(val_ae_loader))

        model = LSTMAutoencoder(
            input_size=len(feature_cols),
            hidden_size=64,
            latent_size=32,
            num_layers=1,
        ).to(device)

        criterion = nn.MSELoss()
        optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

        total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
        print(model)
        print(f"Total trainable parameters: {total_params:,}")

        best_val_loss    = np.inf
        patience_counter = 0
        train_losses     = []
        val_losses       = []

        for epoch in range(1, EPOCHS + 1):

            model.train()
            epoch_train_losses = []

            for batch_x in train_loader:
                batch_x = batch_x.to(device)
                optimizer.zero_grad()
                reconstructed = model(batch_x)
                loss          = criterion(reconstructed, batch_x)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=MAX_GRAD_NORM)
                optimizer.step()
                epoch_train_losses.append(loss.item())

            epoch_train_loss = float(np.mean(epoch_train_losses))
            train_losses.append(epoch_train_loss)

            model.eval()
            epoch_val_losses = []

            with torch.no_grad():
                for batch_x in val_ae_loader:
                    batch_x  = batch_x.to(device)
                    val_loss = criterion(model(batch_x), batch_x)
                    epoch_val_losses.append(val_loss.item())

            epoch_val_loss = float(np.mean(epoch_val_losses))
            val_losses.append(epoch_val_loss)

            gap = epoch_val_loss - epoch_train_loss
            print(
                f"Epoch {epoch:03d} | "
                f"Train: {epoch_train_loss:.6f} | "
                f"Val: {epoch_val_loss:.6f} | "
                f"Gap: {gap:+.6f}"
            )

            if epoch_val_loss < best_val_loss - MIN_DELTA:
                best_val_loss    = epoch_val_loss
                patience_counter = 0
                torch.save(model.state_dict(), model_path)
                print(f"  --> Best model saved (val loss: {best_val_loss:.6f})")
            else:
                patience_counter += 1

            if patience_counter >= PATIENCE:
                print(f"Early stopping at epoch {epoch} (patience={PATIENCE}).")
                break

        print("\nTraining completed.")
        print(f"Best validation loss : {best_val_loss:.6f}")
        print(f"Final train/val gap  : {val_losses[-1] - train_losses[-1]:+.6f}")

        history_df = pd.DataFrame({
            "epoch":      range(1, len(train_losses) + 1),
            "train_loss": train_losses,
            "val_loss":   val_losses,
        })
        history_df.to_csv(history_path, index=False)

        save_training_curve(train_losses, val_losses, PLOT_DIR, SPLIT_NAME)

        assert model_path.exists(), f"Best model not saved: {model_path}"
        model.load_state_dict(
            torch.load(model_path, map_location=device, weights_only=True)
        )
        model.to(device)
        model.eval()
        print("Best model loaded.")

        print("\nScoring train set...")
        train_scored = score_dataframe_with_autoencoder(
            train_sorted, feature_cols, SEQ_LEN, STRIDE_SCORE, model, BATCH_SIZE, device,
        )

        print("\nScoring validation set...")
        val_scored = score_dataframe_with_autoencoder(
            val_sorted, feature_cols, SEQ_LEN, STRIDE_SCORE, model, BATCH_SIZE, device,
        )

        print("\nScoring test set...")
        test_scored = score_dataframe_with_autoencoder(
            test_sorted, feature_cols, SEQ_LEN, STRIDE_SCORE, model, BATCH_SIZE, device,
        )

        train_normal_scores = train_scored.loc[
            train_scored["dataset_label"] == "normal", "anomaly_score"
        ].dropna()
        assert len(train_normal_scores) > 0

        threshold = float(np.percentile(train_normal_scores, 95))
        print(f"\nAnomaly threshold : {threshold:.6f}")
        print(f"Computed from     : {len(train_normal_scores):,} normal training rows")

        for df in [train_scored, val_scored, test_scored]:
            df["anomaly_flag"] = (df["anomaly_score"] > threshold).astype(int)

        for split_label, df in [
            ("Train",      train_scored),
            ("Validation", val_scored),
            ("Test",       test_scored),
        ]:
            print(f"\n{split_label} flagged percentage by label:")
            print(
                (df.groupby("dataset_label")["anomaly_flag"].mean() * 100)
                .round(2)
                .to_string()
            )

        train_out = restore_and_save(train_scored, train_save_path)
        val_out   = restore_and_save(val_scored,   val_save_path)
        test_out  = restore_and_save(test_scored,  test_save_path)

        train_flight_summary = create_flight_summary(train_out, SPLIT_NAME, "Train",      threshold)
        val_flight_summary   = create_flight_summary(val_out,   SPLIT_NAME, "Validation", threshold)
        test_flight_summary  = create_flight_summary(test_out,  SPLIT_NAME, "Test",       threshold)

        train_flight_summary.to_csv(train_flight_summary_path, index=False)
        val_flight_summary.to_csv(val_flight_summary_path,     index=False)
        test_flight_summary.to_csv(test_flight_summary_path,   index=False)

        row_summary = pd.concat([
            create_row_summary(train_out, SPLIT_NAME, "Train"),
            create_row_summary(val_out,   SPLIT_NAME, "Validation"),
            create_row_summary(test_out,  SPLIT_NAME, "Test"),
        ], ignore_index=True)
        row_summary.to_csv(row_summary_path, index=False)

        save_distribution_plot(train_out, val_out, test_out, threshold, PLOT_DIR, SPLIT_NAME)
        save_flight_bar_plot(val_flight_summary,  PLOT_DIR, SPLIT_NAME, "Validation")
        save_flight_bar_plot(test_flight_summary, PLOT_DIR, SPLIT_NAME, "Test")
        save_roc_plot(val_flight_summary,  PLOT_DIR, SPLIT_NAME, "Validation")
        save_roc_plot(test_flight_summary, PLOT_DIR, SPLIT_NAME, "Test")

        del model
        del train_df, val_df, test_df
        del train_sorted, val_sorted, test_sorted
        del train_scored, val_scored, test_scored

        gc.collect()
        if device.type == "mps":    torch.mps.empty_cache()
        elif device.type == "cuda": torch.cuda.empty_cache()

    # Metrics — run whether we trained fresh or loaded from cache
    split_metrics = []
    for data_split, flight_summary in [
        ("Train",      train_flight_summary),
        ("Validation", val_flight_summary),
        ("Test",       test_flight_summary),
    ]:
        metrics = compute_flight_metrics(flight_summary, SPLIT_NAME, data_split)
        metrics["threshold"] = threshold
        split_metrics.append(metrics)

    split_metrics_df = pd.DataFrame(split_metrics)
    split_metrics_df.to_csv(metrics_path, index=False)

    ALL_METRICS.append(split_metrics_df)
    ALL_ROW_SUMMARIES.append(pd.read_csv(row_summary_path, low_memory=False))
    ALL_FLIGHT_SUMMARIES.extend([train_flight_summary, val_flight_summary, test_flight_summary])

    print("\nSplit metrics:")
    print(split_metrics_df.to_string(index=False))

    test_row = split_metrics_df[split_metrics_df["data_split"] == "Test"].iloc[0]
    print(
        f"\nFINISHED {SPLIT_NAME} | "
        f"Test AUROC={test_row['flight_auroc']:.4f} | "
        f"Test separation={test_row['separation_ratio']:.2f}x"
    )


STARTING SPLIT: split_01_seed_11
Loaded files:
  Train: train_no_dr_scaled_v5_voltage_random10_new_features_split_01_seed_11.parquet
  Val  : val_no_dr_scaled_v5_voltage_random10_new_features_split_01_seed_11.parquet
  Test : test_no_dr_scaled_v5_voltage_random10_new_features_split_01_seed_11.parquet
Train: (698993, 19)
Val  : (133277, 19)
Test : (134655, 19)
Data validation passed.
Clean normal training windows   : 24,817
Clean normal validation windows : 4,686
Train batches     : 388
Validation batches: 74
LSTMAutoencoder(
  (encoder_lstm): LSTM(16, 64, batch_first=True)
  (encoder_fc): Linear(in_features=64, out_features=32, bias=True)
  (decoder_input_fc): Linear(in_features=32, out_features=64, bias=True)
  (decoder_h0_fc): Linear(in_features=32, out_features=64, bias=True)
  (decoder_c0_fc): Linear(in_features=32, out_features=64, bias=True)
  (decoder_lstm): LSTM(64, 64, batch_first=True)
  (output_layer): Linear(in_features=64, out_features=16, bias=True)
)
Total trainable par

/var/folders/dg/72q0h6ss3xq0zf641p8b9gnm0000gn/T/ipykernel_57576/2200335034.py:69: RuntimeWarning: invalid value encountered in divide
  anomaly_score = np.where(score_count > 0, score_sum / score_count, np.nan)


  Uncovered rows: 493 — filling with ffill/bfill within flights

Scoring validation set...
  Scoring windows: 13,092


/var/folders/dg/72q0h6ss3xq0zf641p8b9gnm0000gn/T/ipykernel_57576/2200335034.py:69: RuntimeWarning: invalid value encountered in divide
  anomaly_score = np.where(score_count > 0, score_sum / score_count, np.nan)


  Uncovered rows: 107 — filling with ffill/bfill within flights

Scoring test set...
  Scoring windows: 13,218


...

  Uncovered rows: 517 — filling with ffill/bfill within flights

Scoring validation set...
  Scoring windows: 12,758


/var/folders/dg/72q0h6ss3xq0zf641p8b9gnm0000gn/T/ipykernel_57576/2200335034.py:69: RuntimeWarning: invalid value encountered in divide
  anomaly_score = np.where(score_count > 0, score_sum / score_count, np.nan)


  Uncovered rows: 100 — filling with ffill/bfill within flights

Scoring test set...
  Scoring windows: 14,946


/var/folders/dg/72q0h6ss3xq0zf641p8b9gnm0000gn/T/ipykernel_57576/2200335034.py:69: RuntimeWarning: invalid value encountered in divide
  anomaly_score = np.where(score_count > 0, score_sum / score_count, np.nan)


  Uncovered rows: 118 — filling with ffill/bfill within flights

Anomaly threshold : 0.201072
Computed from     : 486,835 normal training rows

Train flagged percentage by label:
dataset_label
abnormal    21.0
normal       5.0

Validation flagged percentage by label:
dataset_label
abnormal    26.42
normal       2.77

Test flagged percentage by label:
dataset_label
abnormal    0.66
normal      4.35
Saved : /Users/arjun/Documents/UAV thesis/uav-thesis/uav_px4_dataset_project/model_outputs/lstm_autoencoder_v5_voltage_random10_new_features/split_10_seed_111/train_no_dr_with_lstm_anomaly_score_v5_voltage_random10_new_features_split_10_seed_111.csv
Shape : (685077, 21)
Missing anomaly_score: 0
Saved : /Users/arjun/Documents/UAV thesis/uav-thesis/uav_px4_dataset_project/model_outputs/lstm_autoencoder_v5_voltage_random10_new_features/split_10_seed_111/val_no_dr_with_lstm_anomaly_score_v5_voltage_random10_new_features_split_10_seed_111.csv
Shape : (129930, 21)
Missing anomaly_score: 0
Saved : /

## 12. Cross-split summary

In [12]:

all_metrics_df = pd.concat(ALL_METRICS, ignore_index=True)
all_metrics_df.to_csv(MASTER_OUTPUT_DIR / "all_splits_metrics.csv", index=False)

all_row_df    = pd.concat(ALL_ROW_SUMMARIES,    ignore_index=True)
all_flight_df = pd.concat(ALL_FLIGHT_SUMMARIES, ignore_index=True)

all_row_df.to_csv(MASTER_OUTPUT_DIR    / "all_splits_row_summary_final.csv",    index=False)
all_flight_df.to_csv(MASTER_OUTPUT_DIR / "all_splits_flight_summary_final.csv", index=False)

test_metrics = all_metrics_df[all_metrics_df["data_split"] == "Test"]

print("\n" + "=" * 70)
print("LSTM AE — 10-SPLIT TEST SUMMARY")
print("=" * 70)

for col, label in [
    ("flight_auroc",    "Flight AUROC"),
    ("flight_auprc",    "Flight AUPRC"),
    ("separation_ratio","Separation ratio"),
]:
    m = test_metrics[col].mean()
    s = test_metrics[col].std()
    print(f"  {label:<25} {m:.4f} ± {s:.4f}")

print("\nPer-split test results:")
print(
    test_metrics[
        ["split", "flight_auroc", "flight_auprc", "separation_ratio", "threshold"]
    ]
    .sort_values("split")
    .round(4)
    .to_string(index=False)
)

print(f"\nAll outputs saved to: {MASTER_OUTPUT_DIR}")
print("Notebook 05 complete. Run 05e next for architecture comparison.")


LSTM AE — 10-SPLIT TEST SUMMARY
  Flight AUROC              0.5391 ± 0.1200
  Flight AUPRC              0.4040 ± 0.1372
  Separation ratio          14.4247 ± 24.8364

Per-split test results:
            split  flight_auroc  flight_auprc  separation_ratio  threshold
 split_01_seed_11        0.5489        0.4169            3.4890     0.1663
 split_02_seed_22        0.3910        0.2402            0.1994     0.1771
 split_03_seed_33        0.4586        0.2920            0.1878     0.1966
 split_04_seed_44        0.4962        0.3927            1.1781     0.1788
 split_05_seed_55        0.6842        0.6732           21.0233     0.1952
 split_06_seed_66        0.5489        0.4170           76.0874     0.1901
 split_07_seed_77        0.6767        0.4666            4.0742     0.1969
 split_08_seed_88        0.6241        0.5470           37.0177     0.2595
 split_09_seed_99        0.6316        0.3678            0.6301     0.1569
split_10_seed_111        0.3308        0.2265            0